# Partitioning and the shuffle: Spark's biggest performance lever

_Apache Spark_

**Partitions are how Spark splits work; the shuffle is when it moves data across the network by key — and it's the first place to look when a job is slow.**

Spark splits every DataFrame into partitions &mdash; independent chunks of rows. A
       partition is the unit of parallelism: one task processes one partition on one core. With 200
       partitions and 50 cores, Spark runs 50 tasks at a time until all 200 are done. Get the partition
       count badly wrong and everything else is moot: too few and most of the cluster sits idle (or a task
       runs out of memory); too many and the driver drowns in tiny-task overhead.

       Most operations are narrow &mdash; each output partition depends on just one input
       partition (a filter, a select, a withColumn). These need no
       data movement, so they're cheap and run inside a single stage.

---

This notebook is a practice scaffold for the **Partitioning and the shuffle: Spark's biggest performance lever** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q pyspark
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PySpark

In [ ]:
# Colab: !pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .master("local[*]")              # use all local cores as "executors"
         .appName("shuffles-partitioning")
         .getOrCreate())

# Small inline dataset: events with a date and a user_id.
rows = [(i, ["2024-01-01", "2024-01-02", "2024-01-03"][i % 3],
         0 if i % 10 < 7 else i)          # user_id 0 is a HOT key (70% of rows) -> skew
        for i in range(3000)]
df = spark.createDataFrame(rows, ["event_id", "date", "user_id"])

# ============================================================
# 1. PARTITIONS are the unit of parallelism — count them
# ============================================================
print("partitions:", df.rdd.getNumPartitions())   # depends on local cores

# ============================================================
# 2. repartition (FULL SHUFFLE, can increase) vs coalesce (NO shuffle, decrease only)
# ============================================================
more = df.repartition(16)                  # shuffle to exactly 16 even partitions
print("after repartition(16):", more.rdd.getNumPartitions())   # 16

fewer = more.coalesce(4)                    # merge down to 4 — no shuffle, may be uneven
print("after coalesce(4):", fewer.rdd.getNumPartitions())      # 4
# coalesce cannot INCREASE: more.coalesce(99) would stay at 16.

# ============================================================
# 3. groupBy TRIGGERS A SHUFFLE — see the Exchange in the plan
# ============================================================
g = df.groupBy("user_id").count()
g.explain()                                # look for: Exchange hashpartitioning(user_id, 200)
# The Exchange node IS the shuffle: rows redistributed across the network by user_id.

# ============================================================
# 4. partitionBy on WRITE -> partition PRUNING on read
# ============================================================
df.write.mode("overwrite").partitionBy("date").parquet("/tmp/events_parquet")
# Layout on disk: /tmp/events_parquet/date=2024-01-01/, date=2024-01-02/, ...

one_day = spark.read.parquet("/tmp/events_parquet").filter(F.col("date") == "2024-01-02")
one_day.explain()        # plan shows PartitionFilters: [date = 2024-01-02] -> reads ONE folder
print("one-day rows:", one_day.count())    # only that date's files are touched (pruning)

# ============================================================
# 5. spark.sql.shuffle.partitions — the default-200 dial
# ============================================================
print("default:", spark.conf.get("spark.sql.shuffle.partitions"))   # 200
spark.conf.set("spark.sql.shuffle.partitions", 8)   # 200 is wasteful for tiny data
g2 = df.groupBy("user_id").count()
print("shuffle partitions now:", g2.rdd.getNumPartitions())          # 8

# ============================================================
# 6. SKEW + the SALTING fix (user_id 0 owns 70% of rows)
# ============================================================
print(df.groupBy("user_id").count().orderBy(F.desc("count")).show(3))  # user_id 0 dominates

salted = (df
    .withColumn("salt", (F.rand(seed=0) * 16).cast("int"))   # spread the hot key 16 ways
    .groupBy("user_id", "salt").agg(F.count("*").alias("c"))  # 1st pass: salted partials
    .groupBy("user_id").agg(F.sum("c").alias("count")))       # 2nd pass: combine partials
print(salted.orderBy(F.desc("count")).show(3))               # same totals, no straggler

# Adaptive Query Execution can handle skew automatically (on by default in Spark 3.2+):
# spark.conf.set("spark.sql.adaptive.enabled", True)
# spark.conf.set("spark.sql.adaptive.skewJoin.enabled", True)

spark.stop()

## Visualize the data & results

_When 3,000 rows are shuffled into 8 partitions, what does an EVEN key distribution look like versus a SKEWED one where a single hot key owns 70% of the rows?_

In [ ]:
import numpy as np

N_PART = 8                 # shuffle into 8 partitions
rng = np.random.RandomState(0)
n = 80_000                 # number of rows to route

# ---- EVEN: keys spread across ~50,000 distinct values ----
even_keys = rng.randint(0, 50_000, size=n)
even_counts = np.bincount(even_keys % N_PART, minlength=N_PART)

# ---- SKEWED: one HOT key (0) owns 70% of rows; rest random ----
skew_keys = np.where(rng.rand(n) < 0.70, 0, rng.randint(1, 50_000, size=n))
skew_counts = np.bincount(skew_keys % N_PART, minlength=N_PART)

print("EVEN :", even_counts.tolist(),
      "max/mean =", round(even_counts.max() / even_counts.mean(), 2))   # ~1.02
print("SKEW :", skew_counts.tolist(),
      "max/mean =", round(skew_counts.max() / skew_counts.mean(), 2))   # ~5.88
print("hot partition share:", round(100 * skew_counts[0] / skew_counts.sum(), 1), "%")  # 73.5
# EVEN : [9934, 9924, 9997, 10220, 10070, 9951, 9949, 9955]  max/mean = 1.02
# SKEW : [58780, 3028, 2988, 3019, 3109, 3004, 3046, 3026]   max/mean = 5.88
# A stage finishes only when its slowest task does, so the skewed p0 sets the runtime.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A Spark stage shows 199 of 200 tasks finished in under a minute, but task 200 has been running for 40 minutes. What is the most likely cause, and what's the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize the pattern: almost all tasks done, one task running far longer. — _A long-tail straggler after a shuffle is the textbook signature of data skew — one partition got most of the rows._
- Identify the shuffle key (the groupBy or join column) and check its value distribution. — _Skew comes from one dominant key — a null, a default id, a sentinel like a logged-out user — that hashes all its rows into one partition._
- Apply a fix: salt the hot key, or enable Adaptive Query Execution skew handling. — _Salting spreads the hot key's rows across many partitions; AQE can detect and split a skewed join partition automatically._

**Answer:** Data skew. The shuffle sends every row for a key to the partition hash(key) % n; one hot key (often null or a default value) owns most of the rows, so its single task runs long after the rest. Adding executors does nothing because the bottleneck is one un-splittable task. Fix by salting the hot key (append a small random suffix so its rows spread across many partitions, then combine the partials) or by enabling AQE (spark.sql.adaptive.enabled), which can detect and split skewed join partitions automatically. Filtering the hot key out and handling it separately also works.

</details>

**Problem 2.** Your job finishes correctly but writes 50,000 tiny Parquet files, one per task, and downstream reads are slow. You want about 50 files. Should you use repartition(50) or coalesce(50), and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note you only need to reduce the number of partitions (from many to 50). — _coalesce can only decrease the count, which is exactly this case._
- Compare cost: repartition(50) triggers a full shuffle; coalesce(50) merges existing partitions on the same executors with no shuffle. — _A shuffle writes to disk and moves data across the network — needless here, since you don't need even sizes, just fewer files._
- Call df.coalesce(50).write.parquet(path). — _It merges the partitions cheaply and writes ~50 files._

**Answer:** Use coalesce(50). You only need to reduce the partition count, and coalesce does that by merging existing partitions on the same executors with no shuffle &mdash; cheap. repartition(50) would give you evenly-sized partitions but pays for a full shuffle you don't need just to control output file count. The one caveat: coalesce can leave partitions uneven and reduces upstream parallelism, so if you need balanced sizes (or to increase the count) use repartition instead. This is the small-files problem; either way, control file count before the write.

</details>

**Problem 3.** You run events.groupBy('country').count().explain() and see an Exchange hashpartitioning(country, 200) node. What is that node, where did the 200 come from, and your data is only a few thousand rows — is 200 a good number?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the Exchange node as the shuffle. — _Every shuffle shows up in the physical plan as an Exchange; it marks a stage boundary and the network/disk redistribution of rows by key._
- Trace the 200 to spark.sql.shuffle.partitions, whose default is 200. — _That config sets how many partitions a shuffle produces; nothing about your data picks it — it's a global default._
- Judge it against the data size: a few thousand rows split into 200 partitions is tens of rows each. — _Hundreds of near-empty tasks add scheduling overhead with no parallelism benefit — too many partitions for tiny data._

**Answer:** The Exchange hashpartitioning(country, 200) node is the shuffle: it redistributes the rows by country so each country's rows land together, and it marks a stage boundary. The 200 comes from spark.sql.shuffle.partitions (its default), not from your data. For a few thousand rows it's far too many &mdash; you get ~200 near-empty tasks whose scheduling overhead dwarfs the work. Lower it with spark.conf.set('spark.sql.shuffle.partitions', 8) (or enable AQE, which can coalesce shuffle partitions down automatically). The default is tuned for large data; small data wants far fewer.

</details>